In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # NSW EV Intelligence Platform - Gold Layer Recommendations
# MAGIC 
# MAGIC **Purpose:** Create business-ready tables with analytics and recommendations
# MAGIC 
# MAGIC **Tables Created:**
# MAGIC - Charger recommendations (nearest, by speed, by location)
# MAGIC - Traffic forecasts (hourly, regional)
# MAGIC - Hazard impact analysis
# MAGIC - Trip demand analysis
# MAGIC - Route recommendations
# MAGIC - Charging stop suggestions
# MAGIC - Regional EV metrics
# MAGIC 
# MAGIC **Output:** Gold tables ready for dashboards and applications

import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import math

# ------------------ Helper: Haversine Distance ------------------
def calculate_distance(lat1, lon1, lat2, lon2):
    """Calculate distance between two coordinates in km using Haversine formula"""
    R = 6371  # Earth radius in km
    lat1_rad = radians(lat1)
    lat2_rad = radians(lat2)
    delta_lat = radians(lat2 - lat1)
    delta_lon = radians(lon2 - lon1)
    
    a = sin(delta_lat/2)**2 + cos(lat1_rad) * cos(lat2_rad) * sin(delta_lon/2)**2
    c = 2 * asin(sqrt(a))
    return R * c

# ------------------ Gold Table: Nearest Charger Recommendations ------------------
@dlt.table(
    name="charger_recommendations_nearest",
    comment="Top 5 nearest EV chargers for each location",
    table_properties={"quality": "gold"}
)
def charger_recommendations_nearest():
    chargers = dlt.read("silver_ev_chargers").filter(col("is_valid_coords") == True)
    sydney_lat, sydney_lon = -33.8688, 151.2093
    
    return (
        chargers
        .withColumn("distance_from_sydney", 
            calculate_distance(lit(sydney_lat), lit(sydney_lon), col("latitude"), col("longitude")))
        .withColumn("rank", row_number().over(Window.orderBy("distance_from_sydney")))
        .filter(col("rank") <= 100)
        .select(
            col("station_id"),
            col("station_name_clean").alias("station_name"),
            col("address_clean").alias("address"),
            col("suburb_clean").alias("suburb"),
            col("latitude"),
            col("longitude"),
            col("charging_speed"),
            col("power_kw_clean").alias("power_kw"),
            col("distance_from_sydney"),
            col("rank").alias("proximity_rank")
        )
    )

# ------------------ Gold Table: Charger Recommendations by Speed ------------------
@dlt.table(
    name="charger_recommendations_by_speed",
    comment="EV chargers categorized by charging speed",
    table_properties={"quality": "gold"}
)
def charger_recommendations_by_speed():
    return (
        dlt.read("silver_ev_chargers")
        .filter(col("is_valid_coords") == True)
        .groupBy("charging_speed", "suburb_clean")
        .agg(
            count("*").alias("charger_count"),
            avg("power_kw_clean").alias("avg_power_kw"),
            max("power_kw_clean").alias("max_power_kw")
        )
        .orderBy(desc("charger_count"))
    )

# ------------------ Gold Table: Charger Recommendations by Location ------------------
@dlt.table(
    name="charger_recommendations_by_location",
    comment="Charger availability by suburb and type",
    table_properties={"quality": "gold"}
)
def charger_recommendations_by_location():
    return (
        dlt.read("silver_ev_chargers")
        .filter(col("is_valid_coords") == True)
        .groupBy("suburb_clean")
        .agg(
            count("*").alias("total_chargers"),
            countDistinct("operator_clean").alias("unique_operators"),
            sum(when(col("charging_speed") == "Fast", 1).otherwise(0)).alias("fast_chargers"),
            sum(when(col("charging_speed") == "Rapid", 1).otherwise(0)).alias("rapid_chargers"),
            avg("power_kw_clean").alias("avg_power_kw")
        )
        .orderBy(desc("total_chargers"))
    )

# ------------------ Gold Table: Traffic Forecast (Hourly) ------------------
@dlt.table(
    name="traffic_forecast_hourly",
    comment="Hourly traffic patterns for forecasting",
    table_properties={"quality": "gold"}
)
def traffic_forecast_hourly():
    return (
        dlt.read("silver_traffic_volume")
        .groupBy("hour", "is_weekend", "is_peak_hour")
        .agg(
            avg("vehicle_count").alias("avg_vehicle_count"),
            avg("avg_speed_kmh").alias("avg_speed"),
            avg("traffic_density_score").alias("avg_density_score"),
            count("*").alias("observation_count")
        )
        .orderBy("hour")
    )

# ------------------ Gold Table: Traffic Forecast (Regional) ------------------
@dlt.table(
    name="traffic_forecast_regional",
    comment="Regional traffic patterns",
    table_properties={"quality": "gold"}
)
def traffic_forecast_regional():
    return (
        dlt.read("silver_traffic_volume")
        .groupBy("location_name_clean")
        .agg(
            avg("vehicle_count").alias("avg_daily_volume"),
            avg("avg_speed_kmh").alias("avg_speed"),
            max("vehicle_count").alias("peak_volume"),
            avg(when(col("is_peak_hour"), col("traffic_density_score")).otherwise(None)).alias("peak_hour_density")
        )
        .orderBy(desc("avg_daily_volume"))
    )

# ------------------ Gold Table: Traffic Hazard Impact ------------------
@dlt.table(
    name="traffic_hazard_impact",
    comment="Analysis of traffic hazards and their impact",
    table_properties={"quality": "gold"}
)
def traffic_hazard_impact():
    return (
        dlt.read("silver_traffic_hazards")
        .filter(col("is_active") == True)
        .groupBy("hazard_type_clean", "severity_level")
        .agg(
            count("*").alias("active_hazards"),
            avg("duration_hours").alias("avg_duration_hours"),
            countDistinct("suburb_clean").alias("affected_suburbs")
        )
        .orderBy(desc("active_hazards"))
    )

# ------------------ Gold Table: Trip Demand Analysis ------------------
@dlt.table(
    name="trip_demand_analysis",
    comment="Trip demand patterns by region and time",
    table_properties={"quality": "gold"}
)
def trip_demand_analysis():
    return (
        dlt.read("silver_traffic_volume")
        .groupBy("location_name_clean", "hour")
        .agg(
            avg("vehicle_count").alias("avg_traffic_volume"),
            avg("traffic_density_score").alias("congestion_score")
        )
        .withColumn("demand_category",
            when(col("avg_traffic_volume") > 1000, "High")
            .when(col("avg_traffic_volume") > 500, "Medium")
            .otherwise("Low"))
        .orderBy(desc("avg_traffic_volume"))
    )

# ------------------ Gold Table: Trip Route Recommendations ------------------
@dlt.table(
    name="trip_route_recommendations",
    comment="Recommended routes based on traffic and hazards",
    table_properties={"quality": "gold"}
)
def trip_route_recommendations():
    return (
        dlt.read("silver_traffic_volume")
        .groupBy("road_name_clean")
        .agg(
            avg("avg_speed_kmh").alias("typical_speed"),
            avg("traffic_density_score").alias("typical_congestion"),
            count("*").alias("data_points")
        )
        .withColumn("route_quality_score",
            (col("typical_speed") / 10) - (col("typical_congestion") * 5))
        .withColumn("recommendation",
            when(col("route_quality_score") > 5, "Highly Recommended")
            .when(col("route_quality_score") > 0, "Recommended")
            .otherwise("Consider Alternatives"))
        .orderBy(desc("route_quality_score"))
    )

# ------------------ Gold Table: Trip Charging Stops ------------------
@dlt.table(
    name="trip_charging_stops",
    comment="Optimal charging stops along routes",
    table_properties={"quality": "gold"}
)
def trip_charging_stops():
    return (
        dlt.read("silver_ev_chargers")
        .filter(col("is_valid_coords") == True)
        .filter(col("charging_speed").isin(["Fast", "Rapid"]))
        .select(
            col("station_id"),
            col("station_name_clean").alias("station_name"),
            col("suburb_clean").alias("suburb"),
            col("latitude"),
            col("longitude"),
            col("charging_speed"),
            col("power_kw_clean").alias("power_kw")
        )
        .withColumn("estimated_charge_time_min",
            when(col("power_kw") >= 100, 20)
            .when(col("power_kw") >= 50, 40)
            .otherwise(60))
        .orderBy(desc("power_kw"))
    )

# ------------------ Gold Table: Regional EV Metrics ------------------
@dlt.table(
    name="regional_ev_metrics",
    comment="Regional EV adoption and infrastructure metrics",
    table_properties={"quality": "gold"}
)
def regional_ev_metrics():
    demographics = dlt.read("silver_regional_demographics")
    chargers = dlt.read("silver_ev_chargers").filter(col("is_valid_coords") == True)
    
    charger_counts = (
        chargers
        .groupBy("suburb_clean")
        .agg(count("*").alias("actual_charger_count"))
    )
    
    return (
        demographics
        .join(charger_counts, demographics.suburb_clean == charger_counts.suburb_clean, "left")
        .select(
            demographics.region_name_clean.alias("region_name"),
            demographics.suburb_clean.alias("suburb"),
            demographics.population,
            demographics.density_category,
            demographics.ev_adoption_rate,
            coalesce(charger_counts.actual_charger_count, lit(0)).alias("charger_count"),
            demographics.chargers_per_1000_people
        )
        .withColumn("infrastructure_score",
            (col("charger_count") * 10 / (col("population") / 1000)))
        .withColumn("readiness_category",
            when(col("infrastructure_score") >= 10, "Excellent")
            .when(col("infrastructure_score") >= 5, "Good")
            .when(col("infrastructure_score") >= 1, "Developing")
            .otherwise("Needs Investment"))
        .orderBy(desc("infrastructure_score"))
    )